# RVC v2 Voice Model Training
**Thesis:** Integrating Speech-to-Text and RVC-Based Voice Conversion into an AI Assistant  
**Author:** Nguyen Hoang Ngoc Bao — 24MSE23204  

Maps to **Section 4.1 Stage 2** of the thesis:
> *Preprocess → F0 extraction (RMVPE) → HuBERT features → Train RVC v2 (~200 epochs) → Build FAISS index → Export .pth + .index*

**Before running:**
1. Set `Runtime > Change runtime type > GPU (T4)`
2. Edit the `CONFIGURATION` cell below (at minimum set `SPEAKER_NAME`)
3. Upload raw `.wav`/`.mp3` recordings to `DATASET_RAW` in your Drive
4. Run all cells top-to-bottom


In [ ]:
# ── A. CONFIGURATION ─────────────────────────────────────────────────────────
# Edit these before running anything else.

SPEAKER_NAME  = "speaker_vi_female"   # identifier used for all output files
SAMPLE_RATE   = 40000                 # RVC v2 standard: 40 kHz
TOTAL_EPOCHS  = 200                   # thesis target (Section 4.1 Stage 2)
BATCH_SIZE    = 8                     # reduce to 4 if CUDA OOM on T4
SAVE_EVERY    = 20                    # checkpoint every N epochs
F0_METHOD     = "rmvpe"               # RMVPE — most accurate per thesis Section 2.3
RVC_VERSION   = "v2"

# Google Drive paths
DRIVE_ROOT     = "/content/drive/MyDrive/rvc_training"
DATASET_RAW    = f"{DRIVE_ROOT}/dataset_{SPEAKER_NAME}/raw"
DATASET_SLICED = f"{DRIVE_ROOT}/dataset_{SPEAKER_NAME}/sliced"
MODELS_DIR     = f"{DRIVE_ROOT}/models"

print("Configuration:")
print(f"  speaker     : {SPEAKER_NAME}")
print(f"  sample_rate : {SAMPLE_RATE} Hz")
print(f"  epochs      : {TOTAL_EPOCHS}")
print(f"  batch_size  : {BATCH_SIZE}")
print(f"  f0_method   : {F0_METHOD}")
print(f"  rvc_version : {RVC_VERSION}")


In [ ]:
# ── B. MOUNT GOOGLE DRIVE ────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount("/content/drive")

for d in [DATASET_RAW, DATASET_SLICED, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Directory layout:")
print(f"  raw audio  →  {DATASET_RAW}")
print(f"  sliced     →  {DATASET_SLICED}")
print(f"  models     →  {MODELS_DIR}")
print()
print("ACTION REQUIRED: upload your raw .wav / .mp3 recordings to:")
print(f"  {DATASET_RAW}")
print("Aim for ~15 min of clean speech (thesis Section 4.1 Stage 1).")


In [ ]:
# ── C. CLONE RVC + INSTALL DEPENDENCIES ─────────────────────────────────────
import os

if not os.path.exists("/content/RVC"):
    !git clone --depth=1 \
        https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI \
        /content/RVC 2>&1 | tail -5
else:
    print("RVC already cloned.")

%cd /content/RVC

!pip install -q -r requirements.txt
!pip install -q pydub librosa soundfile

print("\n✓  Dependencies installed.")


In [ ]:
# ── D. GPU CHECK ─────────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU (T4) and reconnect."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓  GPU  : {gpu_name}")
print(f"   VRAM : {vram_gb:.1f} GB")
if vram_gb < 10:
    print(f"⚠️  Low VRAM — consider reducing BATCH_SIZE to 4.")


In [ ]:
# ── E. DOWNLOAD PRETRAINED V2 MODELS ─────────────────────────────────────────
# RVC v2 requires pretrained Generator/Discriminator (40 kHz) + HuBERT + RMVPE
import os, urllib.request

HF_BASE = "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main"

ASSETS = {
    "assets/pretrained_v2/f0G40k.pth" : f"{HF_BASE}/pretrained_v2/f0G40k.pth",
    "assets/pretrained_v2/f0D40k.pth" : f"{HF_BASE}/pretrained_v2/f0D40k.pth",
    "assets/hubert/hubert_base.pt"     : f"{HF_BASE}/hubert_base.pt",
    "assets/rmvpe/rmvpe.pt"            : f"{HF_BASE}/rmvpe.pt",
}

for dest, url in ASSETS.items():
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.exists(dest):
        print(f"  ✓  {os.path.basename(dest)} (cached)")
    else:
        print(f"  Downloading {os.path.basename(dest)} …")
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / 1e6
        print(f"  ✓  {os.path.basename(dest)}  ({size_mb:.0f} MB)")

print("\n✓  All pretrained assets ready.")


In [ ]:
# ── F. DATASET PREPARATION — slice into 3-8 s segments, normalize ─────────────
# Thesis Section 4.1 Stage 1: "record/clean ~15 min, slice into 3-8 s segments,
# normalise to 16 kHz mono" (we use 40 kHz for RVC v2).
import glob, os
from pydub import AudioSegment
from pydub.silence import split_on_silence


def slice_and_normalize(src: str, out_dir: str,
                        sr: int = SAMPLE_RATE,
                        min_ms: int = 3000,
                        max_ms: int = 8000) -> int:
    audio = AudioSegment.from_file(src)
    audio = audio.set_frame_rate(sr).set_channels(1)
    # Normalize to -20 dBFS
    audio = audio.apply_gain(-20.0 - audio.dBFS)

    chunks = split_on_silence(
        audio,
        min_silence_len=300,
        silence_thresh=audio.dBFS - 16,
        keep_silence=150,
    )

    stem = os.path.splitext(os.path.basename(src))[0]
    saved, buf = 0, AudioSegment.empty()
    for chunk in chunks:
        buf += chunk
        while len(buf) >= min_ms:
            seg = buf[:max_ms]
            buf = buf[max_ms:]
            out = os.path.join(out_dir, f"{stem}_{saved:04d}.wav")
            seg.export(out, format="wav")
            saved += 1
    if len(buf) >= min_ms:
        out = os.path.join(out_dir, f"{stem}_{saved:04d}.wav")
        buf.export(out, format="wav")
        saved += 1
    return saved


sources = (glob.glob(os.path.join(DATASET_RAW, "*.wav")) +
           glob.glob(os.path.join(DATASET_RAW, "*.mp3")) +
           glob.glob(os.path.join(DATASET_RAW, "*.m4a")) +
           glob.glob(os.path.join(DATASET_RAW, "*.flac")))

if not sources:
    print(f"⚠️  No audio files found in:\n    {DATASET_RAW}")
    print("Upload .wav / .mp3 / .m4a files there, then re-run this cell.")
else:
    total_segs = 0
    for f in sources:
        n = slice_and_normalize(f, DATASET_SLICED)
        print(f"  {os.path.basename(f):45s} → {n:3d} segments")
        total_segs += n

    dur_est = total_segs * 5.5 / 60   # ~5.5 s average segment
    print(f"\n✓  {total_segs} segments  (~{dur_est:.1f} min estimated)")
    if total_segs < 50:
        print("⚠️  Aim for at least 50 segments (~10–15 min) for acceptable RVC quality.")
    else:
        print("✓  Dataset size looks good.")


In [ ]:
# ── G. PREPROCESS — RVC trainset_preprocess_pipeline ─────────────────────────
import subprocess, sys, os

os.chdir("/content/RVC")
EXP_DIR = f"logs/{SPEAKER_NAME}"
os.makedirs(EXP_DIR, exist_ok=True)

cmd = [
    sys.executable, "trainset_preprocess_pipeline_print.py",
    DATASET_SLICED,
    str(SAMPLE_RATE),
    "4",          # n_cpu
    EXP_DIR,
    "False",      # noparallel
    "3.7",        # silence trim threshold dB
]

print("Preprocessing …")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Preprocessing failed — check output above.")
print("✓  Preprocessing complete.")


In [ ]:
# ── H. F0 EXTRACTION — RMVPE ─────────────────────────────────────────────────
# Thesis Section 4.1 Stage 2: "F0 extraction with RMVPE"
# RMVPE is the most accurate F0 estimator in the RVC ecosystem (Section 2.3).
import subprocess, sys

cmd = [
    sys.executable, "extract_f0_print.py",
    EXP_DIR,
    "4",          # n_process
    F0_METHOD,    # rmvpe
]

print(f"Extracting F0 with {F0_METHOD} …")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-800:])
    raise RuntimeError("F0 extraction failed.")
print("✓  F0 extraction complete.")


In [ ]:
# ── I. HUBERT FEATURE EXTRACTION ─────────────────────────────────────────────
# Extracts 768-dim HuBERT features (v2). These feed the FAISS retrieval index.
import subprocess, sys

cmd = [
    sys.executable, "extract_feature_print.py",
    "cuda:0",     # device
    "1",          # n_part
    "0",          # i_part
    "0",          # i_gpu
    EXP_DIR,
    RVC_VERSION,  # v2
]

print("Extracting HuBERT features (v2, 768-dim) …")
result = subprocess.run(cmd, capture_output=True, text=True)
if result.stdout:
    print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-800:])
    raise RuntimeError("Feature extraction failed.")
print("✓  HuBERT feature extraction complete.")


In [ ]:
# ── J. GENERATE TRAINING FILELISTS ───────────────────────────────────────────
import glob, os

gt_wavs_dir  = os.path.join(EXP_DIR, "0_gt_wavs")
feature_dir  = os.path.join(EXP_DIR, "3_feature768")  # v2: 768-dim
f0_dir       = os.path.join(EXP_DIR, "2a_f0")
f0nsf_dir    = os.path.join(EXP_DIR, "2b-f0nsf")

wav_files  = sorted(glob.glob(os.path.join(gt_wavs_dir, "*.wav")))
feat_files = sorted(glob.glob(os.path.join(feature_dir, "*.npy")))

wav_map  = {os.path.splitext(os.path.basename(w))[0]: w for w in wav_files}
feat_map = {os.path.splitext(os.path.basename(f))[0]: f for f in feat_files}

common = sorted(set(wav_map) & set(feat_map))
if not common:
    raise RuntimeError(
        "No matching wav/feature pairs found.\n"
        f"  gt_wavs_dir  : {gt_wavs_dir}  ({len(wav_files)} files)\n"
        f"  feature_dir  : {feature_dir}  ({len(feat_files)} files)\n"
        "Check preprocessing and feature extraction steps."
    )

lines = []
for stem in common:
    wav_path   = wav_map[stem]
    feat_path  = feat_map[stem]
    f0_path    = os.path.join(f0_dir,    stem + ".wav.npy")
    f0nsf_path = os.path.join(f0nsf_dir, stem + ".wav.npy")
    spk_id     = "0"   # single speaker
    lines.append(f"{wav_path}|{feat_path}|{f0_path}|{f0nsf_path}|{spk_id}")

filelist_path = os.path.join(EXP_DIR, "filelist.txt")
with open(filelist_path, "w") as fh:
    fh.write("\n".join(lines))

print(f"✓  Filelist: {len(lines)} entries → {filelist_path}")


In [ ]:
# ── K. TRAIN RVC v2 ───────────────────────────────────────────────────────────
# Thesis Section 4.1 Stage 2: "train RVC v2 for ~200 epochs"
# Technique T2 (Section 6.3): RVC inference offloaded to Colab T4 GPU.
import subprocess, sys

PRETRAIN_G = "assets/pretrained_v2/f0G40k.pth"
PRETRAIN_D = "assets/pretrained_v2/f0D40k.pth"

cmd = [
    sys.executable, "train.py",
    "-e",  SPEAKER_NAME,
    "-sr", "40k",
    "-f0", "1",              # use F0 conditioning
    "-bs", str(BATCH_SIZE),
    "-g",  "0",              # GPU 0
    "-te", str(TOTAL_EPOCHS),
    "-se", str(SAVE_EVERY),
    "-pg", PRETRAIN_G,
    "-pd", PRETRAIN_D,
    "-l",  "1",              # save latest checkpoint only
    "-c",  "0",              # no GPU cache (saves VRAM)
    "-sw", "0",              # don't save weights every epoch (saves Drive space)
    "-v",  RVC_VERSION,      # v2
]

print("Training command:")
print("  " + " ".join(cmd))
print(f"\nTraining {TOTAL_EPOCHS} epochs | batch={BATCH_SIZE} | GPU: T4")
print("Expected duration: ~30–60 min for 200 epochs on T4.")
print("-" * 60)

result = subprocess.run(cmd)   # live stdout/stderr
if result.returncode != 0:
    raise RuntimeError("Training failed — check output above.")
print("\n✓  Training complete.")


In [ ]:
# ── L. BUILD FAISS RETRIEVAL INDEX ───────────────────────────────────────────
# Thesis Section 4.1 Stage 2: "build the FAISS retrieval index, export .index"
# The index stores HuBERT features of the training set; at inference time RVC
# retrieves the nearest neighbours to inject target-speaker timbre.
import glob, os
import numpy as np
import faiss

feature_dir = os.path.join(EXP_DIR, "3_feature768")
feat_files  = sorted(glob.glob(os.path.join(feature_dir, "*.npy")))

if not feat_files:
    raise FileNotFoundError(
        f"No .npy feature files in {feature_dir}\n"
        "Re-run the HuBERT feature extraction cell."
    )

print(f"Loading {len(feat_files)} feature files …")
features = np.concatenate(
    [np.load(f) for f in feat_files], axis=0
).astype("float32")
print(f"  Feature matrix: {features.shape}  (dim={features.shape[1]})")

# IVF size: RVC convention is min(16*sqrt(N), N)
n_ivf = min(int(16 * np.sqrt(len(features))), len(features) // 39 + 1)
n_ivf = max(n_ivf, 4)
print(f"  Building IVF{n_ivf},Flat index …")

index = faiss.index_factory(features.shape[1], f"IVF{n_ivf},Flat")
index.train(features)
index.add(features)

index_path = os.path.join(EXP_DIR, f"{SPEAKER_NAME}.index")
faiss.write_index(index, index_path)

size_mb = os.path.getsize(index_path) / 1e6
print(f"✓  FAISS index saved: {index_path}  ({size_mb:.1f} MB)")
print(f"   Vectors indexed: {index.ntotal}")


In [ ]:
# ── M. EXPORT MODEL TO GOOGLE DRIVE ──────────────────────────────────────────
# Final deliverable: <speaker>.pth  +  <speaker>.index  in MODELS_DIR
import glob, os, shutil

def find_latest(pattern):
    files = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    return files[0] if files else None

# Look for the speaker .pth or the generator G_<step>.pth
pth_path = (
    find_latest(f"{EXP_DIR}/{SPEAKER_NAME}_*.pth") or
    find_latest(f"{EXP_DIR}/G_*.pth")
)
index_path = os.path.join(EXP_DIR, f"{SPEAKER_NAME}.index")

if not pth_path:
    raise FileNotFoundError("No .pth checkpoint found — did training finish?")
if not os.path.exists(index_path):
    raise FileNotFoundError(f"Index not found: {index_path}")

out_pth   = os.path.join(MODELS_DIR, f"{SPEAKER_NAME}.pth")
out_index = os.path.join(MODELS_DIR, f"{SPEAKER_NAME}.index")

shutil.copy2(pth_path,   out_pth)
shutil.copy2(index_path, out_index)

print("✓  Exported to Google Drive:")
print(f"   {out_pth}    ({os.path.getsize(out_pth)/1e6:.1f} MB)")
print(f"   {out_index}  ({os.path.getsize(out_index)/1e6:.1f} MB)")
print()
print("Next step: open serve_rvc.ipynb to start the inference server.")
